# Kaggle runner: noise study at the main settings, then the final tables

Runs only `05_noise_hardware/01b_noise_study_aer_frozen.ipynb` (about 1 hour on Kaggle CPU) and `06_analysis/01_final_tables_and_falsifier.ipynb`.
Setup: add the updated Project dataset as input, CPU, Internet on, then Save Version -> Save & Run All (Commit).

In [ ]:
!pip install -q "qiskit>=2.1,<3" "qiskit-aer>=0.17" "qiskit-ibm-runtime>=0.40" pylatexenc

In [ ]:
import os
import shutil
import subprocess
import time
import zipfile
from pathlib import Path

source = next(path.parent for path in Path("/kaggle/input").rglob("qsvm_rc") if path.is_dir())
WORK = Path("/kaggle/working/Project")
if not WORK.exists():
    shutil.copytree(source, WORK)
print("Source:", source, "| CPU cores:", os.cpu_count())

# Stop early if the new notebook was not pasted into the uploaded project
frozen_notebook = WORK / "05_noise_hardware" / "01b_noise_study_aer_frozen.ipynb"
assert frozen_notebook.exists(), "01b_noise_study_aer_frozen.ipynb is missing from the uploaded Project"
print("best_configs.json:", (WORK / "results" / "3x2" / "qsvm" / "best_configs.json").read_text())

In [ ]:
def run_notebook(relative_path):
    """Execute one project notebook in place, from its own folder, with no time limit."""
    path = WORK / relative_path
    start = time.time()
    result = subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace",
                             "--ExecutePreprocessor.timeout=-1", "--ExecutePreprocessor.kernel_name=python3",
                             path.name],
                            cwd = path.parent, capture_output = True, text = True)
    minutes = (time.time() - start) / 60
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError(f"{relative_path} failed after {minutes:.1f} min, see the error above")
    print(f"OK  {relative_path}  ({minutes:.1f} min)")

In [ ]:
run_notebook("05_noise_hardware/01b_noise_study_aer_frozen.ipynb")

In [ ]:
run_notebook("06_analysis/01_final_tables_and_falsifier.ipynb")

In [ ]:
# Only what this run changed: the two executed notebooks, the new noise folder, final tables and figures
archive = Path("/kaggle/working/noise_frozen_outputs.zip")
folders = [WORK / "results" / "3x2" / "noise_frozen", WORK / "results" / "final_tables", WORK / "results" / "figures"]
files = [WORK / "05_noise_hardware" / "01b_noise_study_aer_frozen.ipynb", WORK / "06_analysis" / "01_final_tables_and_falsifier.ipynb"]
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for file in files + [item for folder in folders for item in folder.rglob("*") if item.is_file()]:
        bundle.write(file, Path("Project") / file.relative_to(WORK))
print(f"{archive} ({archive.stat().st_size / 1e6:.1f} MB)")